# Train an instrument classifier on SID-RAS (Colab)

SID-RAS has no bounding boxes -- just one folder per instrument. That makes
it a whole-image classification dataset, not a detection one, so this trains
a **separate** YOLOv8 classifier model, independent of the AVOS YOLOv8
*detector*.

**Before running:** Runtime menu -> Change runtime type -> GPU.

## 1. Confirm a GPU is attached

In [ ]:
!nvidia-smi

## 2. Clone the repo

In [ ]:
!git clone https://github.com/cxia0024/hypospadias-object-detection.git
%cd hypospadias-object-detection
!git checkout claude/surgical-phase-recognition-wqrp7r

## 3. Install dependencies

In [ ]:
!pip install -q -r requirements.txt

## 4. Sanity check: run the unit tests

In [ ]:
!python -m pytest tests/ -q

## 5. Mount Google Drive

SID-RAS should already be on Drive as one folder per instrument, all images
unsplit (e.g. `sidras/bovie/*.jpg`, `sidras/forceps/*.jpg`, ...).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 6. Split into train/val

Stratified per class -- every instrument class gets its own `val_fraction`
applied independently, so a rare class isn't accidentally left with zero
val images by a single global shuffle. Reproducible via `--seed` /
the `seed=` argument.

In [ ]:
import sys
sys.path.insert(0, "src")
from instrument_classifier.train import split_classification_folders, validate_classification_data

DRIVE_ROOT = "/content/drive/MyDrive/hypospadias"  # <-- change to your Drive folder
SIDRAS_SOURCE = f"{DRIVE_ROOT}/data/sidras"          # <-- change to your unsplit SID-RAS folder
SIDRAS_SPLIT = f"{DRIVE_ROOT}/data/sidras_split"

split_classification_folders(SIDRAS_SOURCE, SIDRAS_SPLIT, val_fraction=0.2, seed=42)
classes = validate_classification_data(SIDRAS_SPLIT)
print(f"OK. {len(classes['train'])} classes: {classes['train']}")

## 7. Train

`yolov8n-cls.pt` is the ImageNet-pretrained starting point for a fast run;
swap to `yolov8s-cls.pt`/`yolov8m-cls.pt` for more capacity if the GPU has
the memory and SID-RAS has enough images per class to benefit from it.

In [ ]:
from instrument_classifier.train import train_classifier

best = train_classifier(
    data=SIDRAS_SPLIT,
    model="yolov8n-cls.pt",
    epochs=100,
    imgsz=224,
    batch=64,
    project="runs/train",
    name="sidras_classifier",
    seed=0,
)
print(f"Best checkpoint: {best}")

## 8. Save the checkpoint to Drive

In [ ]:
import shutil
from pathlib import Path

out_path = Path(f"{DRIVE_ROOT}/models/sidras_classifier_best.pt")
out_path.parent.mkdir(parents=True, exist_ok=True)
shutil.copy(best, out_path)
print(f"Saved to {out_path}")